# EfficientNet-B3 + Test-Time Augmentation: Maximum Accuracy

MH2's best model was VGG16 Pruned 70% at **95.88%**. This notebook attempts to beat that ceiling using two powerful techniques:

1. **EfficientNet-B3** — modern architecture (2019) with compound scaling. 12M params vs VGG16's 134M, yet better ImageNet accuracy (81.6% vs 71.6%)
2. **Test-Time Augmentation (TTA)** — free accuracy boost at inference by averaging predictions over multiple augmented views of the same image
3. **Label smoothing** — soft targets to handle ambiguous Cloudy↔Shine boundaries
4. **Cosine annealing LR** — smooth learning rate decay for better convergence

**Target:** Break the 96% barrier. **Run on Google Colab with T4 GPU.**

In [0]:
!wget -q http://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/Multi-classWeatherDataset.zip
!unzip -qq -o Multi-classWeatherDataset.zip

import torch
from torch import nn, optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import time, copy, os, itertools
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}")

In [0]:
# ==================== Transforms ====================
train_transform = transforms.Compose([
    transforms.Resize((300, 300)),  # EfficientNet-B3 optimal input size
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ==================== Dataset (same split as MH1-MH5) ====================
base_dataset = datasets.ImageFolder(root='Multi-class Weather Dataset')
class_names = base_dataset.classes
print(f"Classes: {class_names}, Total: {len(base_dataset)}")

train_size = int(0.70 * len(base_dataset))
val_size = int(0.15 * len(base_dataset))
test_size = len(base_dataset) - train_size - val_size

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, ds, indices, tfm):
        self.samples = [(ds.samples[i][0], ds.samples[i][1]) for i in indices]
        self.transform = tfm
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

g = torch.Generator().manual_seed(42)
idx = torch.randperm(len(base_dataset), generator=g).tolist()
train_loader = torch.utils.data.DataLoader(
    TransformSubset(base_dataset, idx[:train_size], train_transform), batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(
    TransformSubset(base_dataset, idx[train_size:train_size+val_size], test_transform), batch_size=32)
test_loader = torch.utils.data.DataLoader(
    TransformSubset(base_dataset, idx[train_size+val_size:], test_transform), batch_size=32)

# Keep raw test dataset for TTA later
test_dataset_raw = TransformSubset(base_dataset, idx[train_size+val_size:], None)
test_dataset_raw.transform = None  # will apply TTA transforms manually

print(f"Train: {train_size}, Val: {val_size}, Test: {test_size}")

## Section 1: EfficientNet-B3 — Why It Should Beat VGG16

**Architecture comparison:**

| Feature | VGG16 (2014) | EfficientNet-B3 (2019) |
|---------|-------------|------------------------|
| Parameters | 134.3M | **12.2M** (11x smaller) |
| ImageNet top-1 | 71.6% | **81.6%** (+10%) |
| Input size | 224×224 | 300×300 (more detail) |
| Key innovation | Stack conv layers | Compound scaling (depth+width+resolution) |
| Building block | Conv + ReLU | MBConv + Squeeze-Excite + Swish |

**Our training strategy (everything MH2 taught us):**
1. **Deeper fine-tuning** — unfreeze last 2 blocks (not just the classifier)
2. **Label smoothing** (0.1) — soft targets for ambiguous Cloudy↔Shine images
3. **Cosine annealing LR** — smooth decay instead of constant rate
4. **Early stopping** with best checkpoint — lesson learned from MH4's oscillation bug
5. **300×300 input** — EfficientNet-B3's optimal resolution (more detail than 224×224)

In [0]:
# ==================== Build EfficientNet-B3 ====================
model = models.efficientnet_b3(weights='IMAGENET1K_V1')

# Freeze all layers first
for p in model.parameters(): p.requires_grad = False

# Unfreeze last 2 blocks of features + classifier
# EfficientNet-B3 has 8 blocks (features[0] to features[7]) + features[8] = head
for p in model.features[6:].parameters(): p.requires_grad = True
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)
model = model.to(device)

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"EfficientNet-B3: {total_p:,} total params, {train_p:,} trainable ({100*train_p/total_p:.1f}%)")

# ==================== Training Setup ====================
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # soft targets

# Discriminative learning rates
params_groups = [
    {'params': model.features[6:].parameters(), 'lr': 0.0001},   # conv blocks: small lr
    {'params': model.classifier.parameters(), 'lr': 0.001},       # classifier: larger lr
]
optimizer = optim.Adam(params_groups)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-6)

# ==================== Training Loop ====================
best_val_loss, best_state, patience_cnt = None, None, 0
train_losses, val_losses, train_accs, val_accs = [], [], [], []

print(f"\nTraining EfficientNet-B3 (label smoothing=0.1, cosine LR)...")
print(f"{'='*65}")
start = time.time()

for epoch in range(25):
    # Train
    model.train()
    rloss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs); loss = criterion(out, labels)
        loss.backward(); optimizer.step()
        rloss += loss.item() * imgs.size(0)
        _, pred = torch.max(out, 1); total += labels.size(0); correct += (pred == labels).sum().item()
    tl, ta = rloss / total, 100 * correct / total
    
    # Validate
    model.eval(); vloss, vc, vt = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs); vloss += criterion(out, labels).item() * imgs.size(0)
            _, pred = torch.max(out, 1); vt += labels.size(0); vc += (pred == labels).sum().item()
    vl, va = vloss / vt, 100 * vc / vt
    
    scheduler.step()
    train_losses.append(tl); val_losses.append(vl); train_accs.append(ta); val_accs.append(va)
    
    # Early stopping with best checkpoint
    if best_val_loss is None or vl < best_val_loss - 0.001:
        best_val_loss, best_state, patience_cnt = vl, copy.deepcopy(model.state_dict()), 0
        marker = ' ← best'
    else:
        patience_cnt += 1; marker = ''
    
    cur_lr = optimizer.param_groups[0]['lr']
    print(f"  Epoch {epoch+1:2d} | Train: {tl:.4f}/{ta:.1f}% | Val: {vl:.4f}/{va:.1f}% | LR: {cur_lr:.6f}{marker}")
    
    if patience_cnt >= 5:
        print(f"  Early stopping at epoch {epoch+1}!"); break

model.load_state_dict(best_state)
train_time = time.time() - start
print(f"\nTraining completed in {train_time:.1f}s")

# ==================== Test accuracy (standard, no TTA) ====================
model.eval(); tc, tt = 0, 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        _, pred = torch.max(model(imgs), 1); tt += labels.size(0); tc += (pred == labels).sum().item()
standard_acc = 100 * tc / tt
print(f"\nEfficientNet-B3 Test Accuracy (standard): {standard_acc:.2f}%")
print(f"MH2 best (VGG16 pruned 70%): 95.88%")
print(f"Difference: {standard_acc - 95.88:+.2f}%")

In [0]:
# ==================== Plot training curves ====================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(train_losses) + 1)

ax1.plot(ep, train_losses, 'b-o', label='Train', markersize=3)
ax1.plot(ep, val_losses, 'r-s', label='Val', markersize=3)
ax1.set_title('Loss (Label Smoothing + Cosine LR)', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(ep, train_accs, 'b-o', label='Train', markersize=3)
ax2.plot(ep, val_accs, 'r-s', label='Val', markersize=3)
ax2.axhline(y=95.88, color='green', linestyle='--', label='MH2 best (95.88%)', alpha=0.7)
ax2.set_title('Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('EfficientNet-B3 Training', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

gap = train_accs[-1] - val_accs[-1]
print(f"Train-Val gap: {gap:.2f}% {'(healthy)' if gap < 5 else '(overfitting!)'}")

## Section 2: Test-Time Augmentation (TTA)

**Normal inference:** 1 image → 1 prediction

**TTA inference:** 1 image → 7 augmented versions → 7 predictions → average → final answer

Our 7 augmented views:
1. **Original** (as-is)
2. **Horizontal flip** (mirror)
3. **Rotate +10°**
4. **Rotate -10°**
5. **Center crop** (zoom in to 90%)
6. **Brightness +20%**
7. **Brightness -20%**

Each view may emphasize different features (cloud texture, sky color, rain patterns). Averaging cancels noise and boosts correct predictions — especially for ambiguous Cloudy↔Shine images.

**Cost:** 7x slower inference. **Benefit:** +0.5-1.5% accuracy for free (no retraining).

In [0]:
# ==================== TTA Transforms ====================
tta_transforms = [
    # 1. Original
    transforms.Compose([
        transforms.Resize((300, 300)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    # 2. Horizontal flip
    transforms.Compose([
        transforms.Resize((300, 300)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    # 3. Rotate +10
    transforms.Compose([
        transforms.Resize((300, 300)), transforms.RandomRotation((10, 10)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    # 4. Rotate -10
    transforms.Compose([
        transforms.Resize((300, 300)), transforms.RandomRotation((-10, -10)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    # 5. Center crop (zoom 90%)
    transforms.Compose([
        transforms.Resize((333, 333)), transforms.CenterCrop(300), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    # 6. Brighter
    transforms.Compose([
        transforms.Resize((300, 300)), transforms.ColorJitter(brightness=(1.2, 1.2)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    # 7. Darker
    transforms.Compose([
        transforms.Resize((300, 300)), transforms.ColorJitter(brightness=(0.8, 0.8)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
]
print(f"TTA: {len(tta_transforms)} augmented views per image")

# ==================== TTA Prediction Function ====================
def predict_with_tta(model, img_pil, tta_transforms):
    """Predict using TTA: average softmax across all augmented views."""
    model.eval()
    all_probs = []
    with torch.no_grad():
        for tfm in tta_transforms:
            img_tensor = tfm(img_pil).unsqueeze(0).to(device)
            output = model(img_tensor)
            probs = F.softmax(output, dim=1)
            all_probs.append(probs)
    
    avg_probs = torch.stack(all_probs).mean(dim=0)  # average across views
    pred = avg_probs.argmax(dim=1).item()
    return pred, avg_probs[0]

# ==================== Evaluate: Standard vs TTA ====================
# Re-evaluate standard (for fair comparison)
model.eval()
standard_preds, tta_preds, true_labels = [], [], []

print("Evaluating test set with TTA (may take ~30 seconds)...")
for i in range(len(test_dataset_raw.samples)):
    path, label = test_dataset_raw.samples[i]
    img = Image.open(path).convert('RGB')
    true_labels.append(label)
    
    # Standard prediction
    with torch.no_grad():
        img_std = tta_transforms[0](img).unsqueeze(0).to(device)  # just original transform
        std_pred = model(img_std).argmax(dim=1).item()
    standard_preds.append(std_pred)
    
    # TTA prediction
    tta_pred, _ = predict_with_tta(model, img, tta_transforms)
    tta_preds.append(tta_pred)

true_labels = np.array(true_labels)
standard_preds = np.array(standard_preds)
tta_preds = np.array(tta_preds)

standard_acc = 100 * (standard_preds == true_labels).mean()
tta_acc = 100 * (tta_preds == true_labels).mean()

print(f"\n{'='*60}")
print(f"{'EFFICIENTNET-B3 RESULTS':^60}")
print(f"{'='*60}")
print(f"Standard (no TTA):  {standard_acc:.2f}%")
print(f"With TTA (7 views): {tta_acc:.2f}%")
print(f"TTA boost:          {tta_acc - standard_acc:+.2f}%")
print(f"{'='*60}")
print(f"MH2 best (VGG16 pruned 70%): 95.88%")
print(f"EfficientNet-B3 + TTA vs MH2: {tta_acc - 95.88:+.2f}%")

# Show which images TTA fixed
std_wrong = standard_preds != true_labels
tta_wrong = tta_preds != true_labels
fixed_by_tta = std_wrong & ~tta_wrong
broken_by_tta = ~std_wrong & tta_wrong
print(f"\nImages fixed by TTA: {fixed_by_tta.sum()}")
print(f"Images broken by TTA: {broken_by_tta.sum()}")
print(f"Net improvement: {fixed_by_tta.sum() - broken_by_tta.sum()} images")

In [0]:
# ==================== Per-class comparison ====================
print(f"\n{'='*65}")
print(f"{'PER-CLASS ACCURACY':^65}")
print(f"{'='*65}")
print(f"{'Class':<12} {'Standard':<14} {'With TTA':<14} {'TTA Boost':<14}")
print(f"{'-'*65}")
for i, cls in enumerate(class_names):
    mask = true_labels == i
    std_cls = 100 * (standard_preds[mask] == true_labels[mask]).mean()
    tta_cls = 100 * (tta_preds[mask] == true_labels[mask]).mean()
    print(f"{cls:<12} {std_cls:<14.1f} {tta_cls:<14.1f} {tta_cls-std_cls:+.1f}%")
print(f"{'='*65}")

# ==================== Confusion Matrices (Standard vs TTA) ====================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title in [(ax1, standard_preds, 'Standard'), (ax2, tta_preds, 'With TTA')]:
    cm = confusion_matrix(true_labels, preds)
    im = ax.imshow(cm, cmap='Blues')
    for r in range(4):
        for c in range(4):
            ax.text(c, r, str(cm[r, c]), ha='center', va='center',
                   fontsize=14, fontweight='bold',
                   color='white' if cm[r, c] > cm.max()/2 else 'black')
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(class_names, fontsize=9)
    ax.set_yticklabels(class_names, fontsize=9)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    acc = 100 * (preds == true_labels).mean()
    ax.set_title(f'{title}: {acc:.2f}%', fontsize=13, fontweight='bold')

plt.suptitle('EfficientNet-B3 Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Section 3: Fair Comparison — Train VGG16 Pruned 70% on Same Data

To ensure a fair comparison, we train the VGG16 pruned 70% model (MH2's champion) using the exact same data and evaluate with TTA. This way, any improvement from EfficientNet-B3 is purely from the architecture, not from different training runs.

In [0]:
import torch.nn.utils.prune as prune

# ==================== Train VGG16 baseline (same as MH4) ====================
# Need 224x224 transforms for VGG16
vgg_train_tfm = transforms.Compose([
    transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
vgg_test_tfm = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
vgg_train_loader = torch.utils.data.DataLoader(
    TransformSubset(base_dataset, idx[:train_size], vgg_train_tfm), batch_size=32, shuffle=True)
vgg_val_loader = torch.utils.data.DataLoader(
    TransformSubset(base_dataset, idx[train_size:train_size+val_size], vgg_test_tfm), batch_size=32)
vgg_test_loader = torch.utils.data.DataLoader(
    TransformSubset(base_dataset, idx[train_size+val_size:], vgg_test_tfm), batch_size=32)

vgg = models.vgg16(weights='IMAGENET1K_V1')
for p in vgg.parameters(): p.requires_grad = False
for p in vgg.features[24:].parameters(): p.requires_grad = True
vgg.classifier[6] = nn.Linear(4096, 4)
vgg = vgg.to(device)

vgg_criterion = nn.CrossEntropyLoss()
vgg_opt_conv = optim.SGD(vgg.features[24:].parameters(), lr=0.0001, momentum=0.9)
vgg_opt_cls = optim.Adam(vgg.classifier[6].parameters(), lr=0.001)
best_vl, best_vs, pat = None, None, 0

print("Training VGG16 baseline (for fair comparison)...")
for epoch in range(20):
    vgg.train()
    for imgs, labels in vgg_train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        vgg_opt_conv.zero_grad(); vgg_opt_cls.zero_grad()
        loss = vgg_criterion(vgg(imgs), labels); loss.backward()
        vgg_opt_conv.step(); vgg_opt_cls.step()
    vgg.eval(); vloss, vc, vt = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in vgg_val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = vgg(imgs); vloss += vgg_criterion(out, labels).item()*imgs.size(0)
            _, p = torch.max(out, 1); vt += labels.size(0); vc += (p==labels).sum().item()
    vl = vloss/vt
    if best_vl is None or vl < best_vl - 0.001:
        best_vl, best_vs, pat = vl, copy.deepcopy(vgg.state_dict()), 0
    else: pat += 1
    if pat >= 3: print(f"  Early stop at epoch {epoch+1}"); vgg.load_state_dict(best_vs); break

# Prune 70% + fine-tune
print("Applying 70% pruning + fine-tuning...")
params_to_prune = [(m,'weight') for _,m in vgg.named_modules() if isinstance(m,(nn.Conv2d,nn.Linear))]
prune.global_unstructured(params_to_prune, pruning_method=prune.L1Unstructured, amount=0.7)
for _,m in vgg.named_modules():
    if isinstance(m,(nn.Conv2d,nn.Linear)):
        try: prune.remove(m,'weight')
        except: pass

for p in vgg.parameters(): p.requires_grad = True
ft_opt = optim.Adam(vgg.parameters(), lr=0.00005)
best_ft_acc, best_ft_state = 0, None
for epoch in range(5):
    vgg.train()
    for imgs, labels in vgg_train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        ft_opt.zero_grad(); loss = vgg_criterion(vgg(imgs), labels); loss.backward(); ft_opt.step()
    vgg.eval(); vc, vt = 0, 0
    with torch.no_grad():
        for imgs, labels in vgg_val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            _, p = torch.max(vgg(imgs), 1); vt += labels.size(0); vc += (p==labels).sum().item()
    va = 100*vc/vt
    if va > best_ft_acc: best_ft_acc = va; best_ft_state = copy.deepcopy(vgg.state_dict())
vgg.load_state_dict(best_ft_state)

# VGG16 standard test accuracy
vgg.eval(); tc, tt = 0, 0
with torch.no_grad():
    for imgs, labels in vgg_test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        _, p = torch.max(vgg(imgs), 1); tt += labels.size(0); tc += (p==labels).sum().item()
vgg_standard = 100*tc/tt

# VGG16 TTA (224x224 transforms)
vgg_tta_transforms = [
    transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((224,224)), transforms.RandomRotation((10,10)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((224,224)), transforms.RandomRotation((-10,-10)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((249,249)), transforms.CenterCrop(224), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((224,224)), transforms.ColorJitter(brightness=(1.2,1.2)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((224,224)), transforms.ColorJitter(brightness=(0.8,0.8)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
]

vgg_tta_preds = []
print("Evaluating VGG16 pruned with TTA...")
for i in range(len(test_dataset_raw.samples)):
    path, label = test_dataset_raw.samples[i]
    img = Image.open(path).convert('RGB')
    pred, _ = predict_with_tta(vgg, img, vgg_tta_transforms)
    vgg_tta_preds.append(pred)
vgg_tta_preds = np.array(vgg_tta_preds)
vgg_tta_acc = 100 * (vgg_tta_preds == true_labels).mean()

print(f"\n{'='*60}")
print(f"{'FINAL COMPARISON':^60}")
print(f"{'='*60}")
print(f"{'Model':<30} {'Standard':<14} {'With TTA':<14}")
print(f"{'-'*60}")
print(f"{'VGG16 Pruned 70%':<30} {vgg_standard:<14.2f} {vgg_tta_acc:<14.2f}")
print(f"{'EfficientNet-B3':<30} {standard_acc:<14.2f} {tta_acc:<14.2f}")
print(f"{'-'*60}")
best_model_name = 'EfficientNet-B3 + TTA' if tta_acc >= vgg_tta_acc else 'VGG16 Pruned + TTA'
best_acc = max(tta_acc, vgg_tta_acc)
print(f"{'BEST':<30} {best_model_name}: {best_acc:.2f}%")
print(f"{'='*60}")

In [0]:
# ==================== Size + Speed Comparison ====================
def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    nonzero = sum((p != 0).sum().item() for p in m.parameters())
    return total, nonzero

eff_total, eff_active = count_params(model)
vgg_total, vgg_active = count_params(vgg)

print(f"\n{'='*65}")
print(f"{'MODEL EFFICIENCY COMPARISON':^65}")
print(f"{'='*65}")
print(f"{'Metric':<25} {'VGG16 Pruned 70%':<20} {'EfficientNet-B3':<20}")
print(f"{'-'*65}")
print(f"{'Total params':<25} {vgg_total:>15,} {eff_total:>15,}")
print(f"{'Active params':<25} {vgg_active:>15,} {eff_active:>15,}")
print(f"{'Standard accuracy':<25} {vgg_standard:>14.2f}% {standard_acc:>14.2f}%")
print(f"{'TTA accuracy':<25} {vgg_tta_acc:>14.2f}% {tta_acc:>14.2f}%")

# Speed comparison
model.eval(); vgg.eval()
runs = 50

# EfficientNet speed
inp_eff = torch.randn(1, 3, 300, 300).to(device)
start = time.time()
with torch.no_grad():
    for _ in range(runs): _ = model(inp_eff)
eff_time = (time.time() - start) / runs * 1000

# VGG16 speed
inp_vgg = torch.randn(1, 3, 224, 224).to(device)
start = time.time()
with torch.no_grad():
    for _ in range(runs): _ = vgg(inp_vgg)
vgg_time = (time.time() - start) / runs * 1000

print(f"{'Inference (ms)':<25} {vgg_time:>14.1f}ms {eff_time:>14.1f}ms")
print(f"{'Inference TTA (ms)':<25} {vgg_time*7:>14.1f}ms {eff_time*7:>14.1f}ms")
print(f"{'='*65}")
print(f"\nEfficientNet-B3 is {vgg_total/eff_total:.1f}x smaller with {'better' if tta_acc > vgg_tta_acc else 'comparable'} accuracy.")

## Observations & Key Learnings

### EfficientNet-B3 vs VGG16 Pruned 70%:
- **Architecture matters:** EfficientNet-B3 (2019) has fundamentally better feature extraction than VGG16 (2014), achieving comparable or better accuracy with 11x fewer parameters
- **Compound scaling works:** Scaling depth, width, and resolution together (EfficientNet's key insight) beats VGG16's brute-force approach of stacking conv layers
- **300×300 input helps:** Higher resolution captures more weather detail than 224×224

### Test-Time Augmentation:
- **No boost on this dataset** — TTA fixed 1 image but broke 1 image (net = 0)
- The 7 misclassified images are genuinely ambiguous — no viewing angle helps
- TTA works best when errors are due to pose/orientation, not fundamental class overlap
- Trade-off: 7x slower inference — not worth it when the bottleneck is data quality, not model quality

### The Dataset Ceiling Proof (the key finding):
- Three completely independent methods all converge on **95.88%**: VGG16 Pruned (MH2), EfficientNet-B3, EfficientNet-B3 + TTA
- VGG16 Pruned is unstable: got 94.12% in this run vs 95.88% in MH2 (pruning creates run-to-run variance)
- **The bottleneck is data quality (ambiguous Cloudy↔Shine photos), not model quality**
- Only way to break 96%: better/more training data, not better architectures

### Label Smoothing:
- Prevents overconfidence on ambiguous images
- Instead of hard targets [1, 0, 0, 0], uses soft targets [0.9, 0.033, 0.033, 0.033]
- Particularly valuable for our dataset where Cloudy↔Shine boundaries are genuinely fuzzy

### Cosine Annealing LR:
- Smooth learning rate decay (high → low → high → low) instead of constant
- Helps find flatter minima = better generalization
- Combined with early stopping = best of both worlds

### Complete Hackathon Pipeline (6 Notebooks):

| Notebook | Focus | Key Achievement |
|----------|-------|-----------------|
| **MH1** | Classification baseline | ResNet18 95.29%, VGG16 94.71% |
| **MH2** | Optimization | Pruned VGG16 95.88% (best classic) |
| **MH3** | Knowledge distillation | MobileNetV2 student (3.4M params) |
| **MH4** | Deployment | ONNX 128MB, live demo + weather API |
| **MH5** | Generation | Conditional GAN creates weather images |
| **MH6** | Maximum accuracy | EfficientNet-B3 + TTA (modern architecture) |

### Techniques Explored Across All Notebooks:
- Transfer learning (FC-only and fine-tuning)
- Data augmentation (and why it hurts FC-only training)
- Early stopping with best checkpoint
- Ensembling (soft voting)
- Knowledge distillation (T=3.0, alpha=0.7)
- Pruning (50%, 70% global L1 unstructured)
- ONNX export + INT8 quantization
- Conditional GAN with pretrained discriminator
- **EfficientNet-B3** (compound scaling architecture)
- **Test-Time Augmentation** (7-view averaging)
- **Label smoothing** (soft targets)
- **Cosine annealing LR** (smooth learning rate schedule)
- **Xavier / Kaiming weight initialization** (discussed)

From transfer learning basics to modern architectures and deployment — a complete ML engineering journey.